In [3]:
import sys
sys.path.insert(0, '../..')

import requests
import json
import time

BASE = "http://localhost:8000"

print("PRE-STREAMLIT API CHECK")
print("=" * 45)

checks = {}

# Health
r = requests.get(f"{BASE}/health")
checks['health'] = r.status_code == 200
print(f"  /health     : "
      f"{'✅' if checks['health'] else '❌'} "
      f"{r.json().get('status')}")

# Recommend
r = requests.post(
    f"{BASE}/recommend",
    json={"user_id": 481, "top_k": 5})
checks['recommend'] = r.status_code == 200
data = r.json()
print(f"  /recommend  : "
      f"{'✅' if checks['recommend'] else '❌'} "
      f"n_recs={data.get('n_recs')} "
      f"cached={data.get('cached')}")

# Feedback
r = requests.post(
    f"{BASE}/feedback",
    json={
        "user_id":  1,
        "movie_id": 356,
        "rating":   4.5,
        "action":   "rate",
    })
checks['feedback'] = r.status_code == 200
print(f"  /feedback   : "
      f"{'✅' if checks['feedback'] else '❌'}")

# Metrics
r = requests.get(f"{BASE}/metrics")
checks['metrics'] = r.status_code == 200
m = r.json()
print(f"  /metrics    : "
      f"{'✅' if checks['metrics'] else '❌'} "
      f"total={m.get('total_requests')}")

# Cache stats
r = requests.get(f"{BASE}/cache/stats")
checks['cache'] = r.status_code == 200
c = r.json()
print(f"  /cache/stats: "
      f"{'✅' if checks['cache'] else '❌'} "
      f"hit_rate={c.get('hit_rate_pct')}%")

all_ok = all(checks.values())
print(f"\n{'✅ All APIs ready for Streamlit' if all_ok else '❌ Fix APIs before running Streamlit'}")

PRE-STREAMLIT API CHECK
  /health     : ✅ healthy
  /recommend  : ✅ n_recs=5 cached=True
  /feedback   : ✅
  /metrics    : ✅ total=3
  /cache/stats: ✅ hit_rate=51.5%

✅ All APIs ready for Streamlit


In [4]:
# Test Movie Data
import pandas as pd
from pathlib import Path

BASE_PATH = Path('../..')
PROC      = BASE_PATH / 'data' / 'processed'

print("MOVIE DATA CHECK")
print("=" * 45)

# Load movies
movies = pd.read_csv(
    PROC / 'movies_master.csv',
    low_memory=False)
movies = movies[
    movies['movieId'].notna()].copy()
movies['movieId'] = \
    movies['movieId'].astype(int)

# Load ratings
ratings = pd.read_csv(
    PROC / 'ratings_cleaned.csv')

print(f"Movies loaded  : {len(movies):,}")
print(f"Ratings loaded : {len(ratings):,}")

# Check columns
print(f"\nMovie columns: {list(movies.columns)}")

# Check for poster paths
poster_cols = [
    c for c in movies.columns
    if 'poster' in c.lower()
    or 'tmdb' in c.lower()
    or 'imdb' in c.lower()]
print(f"Poster cols  : {poster_cols}")

# Sample user history
uid = ratings['userId'].value_counts()\
    .index[0]
user_movies = ratings[
    ratings['userId'] == uid
].merge(
    movies[['movieId', 'title']],
    on='movieId', how='left')\
    .sort_values('rating', ascending=False)

print(f"\nSample user {uid} top 5:")
for _, row in user_movies.head(5)\
        .iterrows():
    print(f"  {row['title'][:40]:<40} "
          f"⭐ {row['rating']}")

MOVIE DATA CHECK
Movies loaded  : 45,454
Ratings loaded : 100,004

Movie columns: ['id', 'title', 'original_title', 'overview', 'tagline', 'genres', 'release_date', 'year', 'original_language', 'budget', 'revenue', 'runtime', 'vote_average', 'vote_count', 'popularity', 'production_companies', 'poster_path', 'imdb_id', 'cast_names', 'director', 'keyword_list', 'movieId', 'tmdbId', 'genres_list']
Poster cols  : ['poster_path', 'imdb_id', 'tmdbId']

Sample user 547 top 5:
  The Beatles: Eight Days a Week - The Tou ⭐ 5.0
  The Treasure of the Sierra Madre         ⭐ 5.0


TypeError: 'float' object is not subscriptable